In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
!pip install accelerate transformers safetensors opencv-python diffusers

In [2]:
from diffusers import ControlNetModel, StableDiffusionXLControlNetPipeline, AutoencoderKL
from diffusers.utils import load_image
from PIL import Image
import torch
import numpy as np
import cv2

# --- Config ---
SKETCH_PATH = "/kaggle/input/datasets/manalhasan1/sketches/Sketch.jpg"  # local path or URL
PROMPT = "a refined 2D illustration of a character, high quality, detailed"
NEGATIVE_PROMPT = "low quality, bad quality, blurry"
OUTPUT_PATH = "output.png"

# --- Load sketch ---
sketch = load_image(SKETCH_PATH)

# --- Preprocess: extract line art using Canny ---
sketch_np = np.array(sketch)
edges = cv2.Canny(sketch_np, 100, 200)
edges = edges[:, :, None]
edges = np.concatenate([edges, edges, edges], axis=2)
control_image = Image.fromarray(edges)

# --- Load MistoLine ControlNet ---
controlnet = ControlNetModel.from_pretrained(
    "TheMistoAI/MistoLine",
    torch_dtype=torch.float16,
    variant="fp16",
)

# --- Load VAE (fp16 fix for SDXL) ---
vae = AutoencoderKL.from_pretrained(
    "madebyollin/sdxl-vae-fp16-fix",
    torch_dtype=torch.float16
)

# --- Load SDXL pipeline with MistoLine ---
pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    controlnet=controlnet,
    vae=vae,
    torch_dtype=torch.float16,
)
pipe.enable_model_cpu_offload()  # saves VRAM

# --- Generate refined image ---
result = pipe(
    prompt=PROMPT,
    negative_prompt=NEGATIVE_PROMPT,
    image=control_image,
    controlnet_conditioning_scale=1.0,  # how strongly sketch is followed
    num_inference_steps=30,
    guidance_scale=7.0,
).images[0]

result.save(OUTPUT_PATH)
print(f"Saved to {OUTPUT_PATH}")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

diffusion_pytorch_model.fp16.safetensors:   0%|          | 0.00/2.50G [00:00<?, ?B/s]

config.json:   0%|          | 0.00/631 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

...

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

Saved to output.png


In [1]:
from diffusers import ControlNetModel, StableDiffusionXLControlNetPipeline, AutoencoderKL, DPMSolverMultistepScheduler
from diffusers.utils import load_image
from PIL import Image
import torch
import numpy as np
import cv2

# --- Config ---
SKETCH_PATH = "/kaggle/input/datasets/manalhasan1/sketches/Sketch.jpg"  # local path or URL
PROMPT = "a refined 2D illustration, high quality of a leafless tree with a bird on a branch"
NEGATIVE_PROMPT = "low quality, bad quality, blurry"
OUTPUT_PATH = "trial2.png"

# --- Load & preprocess sketch ---
sketch = load_image(SKETCH_PATH)
sketch_np = np.array(sketch)
edges = cv2.Canny(sketch_np, 100, 200)
edges = edges[:, :, None]
edges = np.concatenate([edges, edges, edges], axis=2)
control_image = Image.fromarray(edges)

# --- Load MistoLine ControlNet ---
controlnet = ControlNetModel.from_pretrained(
    "TheMistoAI/MistoLine",
    torch_dtype=torch.float16,
    variant="fp16",
)

# --- Load VAE ---
vae = AutoencoderKL.from_pretrained(
    "madebyollin/sdxl-vae-fp16-fix",
    torch_dtype=torch.float16
)

# --- Load SDXL pipeline ---
pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    controlnet=controlnet,
    vae=vae,
    torch_dtype=torch.float16,
)

# --- Set DPM++ 2M SDE Karras scheduler ---
pipe.scheduler = DPMSolverMultistepScheduler.from_config(
    pipe.scheduler.config,
    algorithm_type="sde-dpmsolver++",   # dpmpp_2m_sde
    use_karras_sigmas=True,             # karras
)

pipe.enable_model_cpu_offload()

# --- ControlNet start/end percent → step numbers ---
total_steps = 30
start_step = int(0.0 * total_steps)   # start_percent: 0.0 → step 0
end_step   = int(0.9 * total_steps)   # end_percent:   0.9 → step 27

# --- Generate ---
result = pipe(
    prompt=PROMPT,
    negative_prompt=NEGATIVE_PROMPT,
    image=control_image,

    # Sampler settings
    num_inference_steps=30,            # steps: 30
    guidance_scale=7.0,                # CFG: 7.0
    strength=0.93,                     # denoise: 0.93

    # ControlNet settings
    controlnet_conditioning_scale=1.0, # controlnet_strength: 1.0
    control_guidance_start=0.0,        # start_percent: 0.0
    control_guidance_end=0.9,          # end_percent: 0.9
).images[0]

result.save(OUTPUT_PATH)
print(f"Saved to {OUTPUT_PATH}")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

Saved to trial2.png


In [2]:
from diffusers import ControlNetModel, StableDiffusionXLControlNetPipeline, AutoencoderKL, DPMSolverMultistepScheduler
from PIL import Image
import torch
import numpy as np
import cv2

# --- Config ---
SKETCH_PATH = "/kaggle/input/datasets/manalhasan1/sketches/Sketch.jpg"  # local path or URL
PROMPT = "a beautiful bare tree with a small bird perched on a branch, 2D flat illustration, clean line art, soft colors, nature artwork, white background, high quality"
NEGATIVE_PROMPT = "low quality, blurry, distorted, chaotic, noisy, extra objects, dark background, photorealistic, 3D"
OUTPUT_PATH = "trial3.png"
IMAGE_SIZE = (768, 1024)

# ----------------------------------------------------------------
# STEP 1: Preprocess — extract cyan lines → black on white
# ----------------------------------------------------------------
sketch = Image.open(SKETCH_PATH).convert("RGB")
sketch = sketch.resize(IMAGE_SIZE, Image.LANCZOS)
sketch_np = np.array(sketch)

r = sketch_np[:, :, 0].astype(int)
g = sketch_np[:, :, 1].astype(int)
b = sketch_np[:, :, 2].astype(int)

# Cyan lines have G and B significantly higher than R
cyan_mask = (g - r) > 15
line_map = np.where(cyan_mask, 0, 255).astype(np.uint8)  # black lines, white bg

control_np = cv2.cvtColor(line_map, cv2.COLOR_GRAY2RGB)
control_image = Image.fromarray(control_np)
control_image.save("debug_control.png")
print("✓ Control image saved — verify debug_control.png looks correct")

# ----------------------------------------------------------------
# STEP 2: Load models
# ----------------------------------------------------------------
print("Loading ControlNet...")
controlnet = ControlNetModel.from_pretrained(
    "TheMistoAI/MistoLine",
    torch_dtype=torch.float16,
    variant="fp16",
)

vae = AutoencoderKL.from_pretrained(
    "madebyollin/sdxl-vae-fp16-fix",
    torch_dtype=torch.float16
)

print("Loading SDXL pipeline...")
pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    controlnet=controlnet,
    vae=vae,
    torch_dtype=torch.float16,
)

# DPM++ 2M SDE Karras
pipe.scheduler = DPMSolverMultistepScheduler.from_config(
    pipe.scheduler.config,
    algorithm_type="sde-dpmsolver++",
    use_karras_sigmas=True,
)

pipe.enable_model_cpu_offload()

# ----------------------------------------------------------------
# STEP 3: Generate  (NO strength= here — that's img2img only)
# ----------------------------------------------------------------
print("Generating...")
generator = torch.Generator().manual_seed(42)

result = pipe(
    prompt=PROMPT,
    negative_prompt=NEGATIVE_PROMPT,
    image=control_image,
    num_inference_steps=30,          # steps: 30
    guidance_scale=7.0,              # CFG: 7.0
    controlnet_conditioning_scale=1.0,  # controlnet_strength: 1.0
    control_guidance_start=0.0,      # start_percent: 0.0
    control_guidance_end=0.9,        # end_percent: 0.9
    width=IMAGE_SIZE[0],
    height=IMAGE_SIZE[1],
    generator=generator,
).images[0]

result.save(OUTPUT_PATH)
print(f"✓ Done! Saved to {OUTPUT_PATH}")

✓ Control image saved — verify debug_control.png looks correct
Loading ControlNet...
Loading SDXL pipeline...


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Generating...


  0%|          | 0/30 [00:00<?, ?it/s]

✓ Done! Saved to trial3.png


Following is the best performing one

In [3]:
from diffusers import ControlNetModel, StableDiffusionXLControlNetPipeline, AutoencoderKL, DPMSolverMultistepScheduler
from PIL import Image
import torch
import numpy as np
import cv2

# --- Config ---
SKETCH_PATH = "/kaggle/input/datasets/manalhasan1/sketches/Sketch.jpg"
OUTPUT_PATH = "trial4.png"
IMAGE_SIZE = (768, 1024)

# Push toward real rendered image, NOT sketch/line art
PROMPT = """a bare winter tree with a small bird perched on a branch, 
digital painting, 2D illustration, soft watercolor style, 
detailed bark texture, warm autumn sky background, 
golden hour lighting, peaceful nature scene, 
highly detailed, professional art, trending on artstation"""

NEGATIVE_PROMPT = """sketch, line art, outline only, drawing, pencil, 
low quality, blurry, distorted, chaotic, watermark, 
signature, jpeg artifacts, oversaturated, flat color, 
no background, white background, monochrome"""

# ----------------------------------------------------------------
# STEP 1: Preprocess sketch — cyan lines → black on white
# ----------------------------------------------------------------
sketch = Image.open(SKETCH_PATH).convert("RGB")
sketch = sketch.resize(IMAGE_SIZE, Image.LANCZOS)
sketch_np = np.array(sketch)

r = sketch_np[:, :, 0].astype(int)
g = sketch_np[:, :, 1].astype(int)

cyan_mask = (g - r) > 15
line_map = np.where(cyan_mask, 0, 255).astype(np.uint8)
control_np = cv2.cvtColor(line_map, cv2.COLOR_GRAY2RGB)
control_image = Image.fromarray(control_np)
control_image.save("debug_control.png")

# ----------------------------------------------------------------
# STEP 2: Load illustration-tuned SDXL model
# ----------------------------------------------------------------
print("Loading ControlNet...")
controlnet = ControlNetModel.from_pretrained(
    "TheMistoAI/MistoLine",
    torch_dtype=torch.float16,
    variant="fp16",
)

vae = AutoencoderKL.from_pretrained(
    "madebyollin/sdxl-vae-fp16-fix",
    torch_dtype=torch.float16
)

print("Loading illustration-tuned SDXL...")
pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
    # Swap from base SDXL → illustration/art focused model
    "Lykon/dreamshaper-xl-1-0",        # ← key change, great for 2D art
    controlnet=controlnet,
    vae=vae,
    torch_dtype=torch.float16,
)

# DPM++ 2M SDE Karras
pipe.scheduler = DPMSolverMultistepScheduler.from_config(
    pipe.scheduler.config,
    algorithm_type="sde-dpmsolver++",
    use_karras_sigmas=True,
)

pipe.enable_model_cpu_offload()

# ----------------------------------------------------------------
# STEP 3: Generate
# ----------------------------------------------------------------
print("Generating...")

# Try a few seeds and pick best result
for seed in [42, 123, 777]:
    generator = torch.Generator().manual_seed(seed)
    result = pipe(
        prompt=PROMPT,
        negative_prompt=NEGATIVE_PROMPT,
        image=control_image,
        num_inference_steps=30,
        guidance_scale=7.0,
        controlnet_conditioning_scale=0.85,  # slightly loosened so it renders scenery
        control_guidance_start=0.0,
        control_guidance_end=0.9,
        width=IMAGE_SIZE[0],
        height=IMAGE_SIZE[1],
        generator=generator,
    ).images[0]

    out = f"output_seed{seed}.png"
    result.save(out)
    print(f"✓ Saved {out}")

Loading ControlNet...
Loading illustration-tuned SDXL...


model_index.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

...

✓ Saved output_seed42.png


  0%|          | 0/30 [00:00<?, ?it/s]

✓ Saved output_seed123.png


  0%|          | 0/30 [00:00<?, ?it/s]

✓ Saved output_seed777.png
